# Полный пайплайн анализа видео

Принимает ссылку на видео и собирает по нему отчёт: субтитры, речь, объекты, классы, описания сцен, постобработку.

Результаты сохраняются в отдельную папку на Drive с названием ролика.
Если ролик уже обрабатывался — пайплайн сообщит об этом и вернёт путь к готовым результатам.

## Установка

In [ ]:
!pip install yt-dlp easyocr ultralytics openai-whisper moviepy rapidfuzz torch torchvision tqdm requests pandas matplotlib -q
!apt-get install -y ffmpeg -q

## Параметры

In [ ]:
VIDEO_URL = 'https://www.youtube.com/watch?v=9SKcLx3GluA'

FRAME_STEP = 30
CONF       = 0.4
WINDOW     = 5
PAUSE      = 3
API_URL    = 'http://92.51.37.40:8000'

## Подготовка папок + проверка идемпотентности

In [ ]:
import subprocess, re, os, sys
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE = '/content/drive/MyDrive/cv-frames'

# получаем название ролика через yt-dlp
title_result = subprocess.run(
    ['yt-dlp', '--print', 'title', '--no-playlist', VIDEO_URL],
    capture_output=True, text=True, timeout=30
)
if title_result.returncode != 0:
    raise RuntimeError(f'не удалось получить название: {title_result.stderr}')

raw_title   = title_result.stdout.strip()
video_title = re.sub(r'[^\w\s\-]', '', raw_title).strip().replace(' ', '_')[:80]

VIDEO_DIR   = f'{BASE_DRIVE}/{video_title}'
FRAMES_DIR  = f'{VIDEO_DIR}/кадры'
RESULTS_DIR = VIDEO_DIR
REPORT_PATH = f'{VIDEO_DIR}/final_report.json'

print(f'название ролика: {raw_title}')
print(f'папка: {VIDEO_DIR}')

# идемпотентность: если отчёт уже есть — выходим
if os.path.exists(REPORT_PATH):
    print()
    print('✓ уже проверяли этот ролик')
    print(f'  результаты здесь: {VIDEO_DIR}')
    print()
    with open(REPORT_PATH, encoding='utf-8') as f:
        import json
        print(json.dumps(json.load(f), ensure_ascii=False, indent=2))
    raise SystemExit('анализ уже выполнен — повторная обработка не нужна')

os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('папки готовы, начинаем обработку')

## ПЗ 2 — Загрузка видео и нарезка кадров

In [ ]:
video_path = f'{VIDEO_DIR}/video.mp4'

!yt-dlp -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best" \
    -o "{video_path}" \
    --no-playlist \
    "{VIDEO_URL}"

if os.path.exists(video_path):
    size_mb = os.path.getsize(video_path) / 1024 / 1024
    print(f'скачано: {size_mb:.1f} MB — {video_path}')
else:
    print('не скачалось, проверьте ссылку')

In [ ]:
import cv2

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print('видео не открылось, проверьте файл')
else:
    fps   = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f'fps: {fps:.1f}, кадров: {total}, длина: {total/fps:.1f} сек')

In [ ]:
from tqdm.notebook import tqdm

frame_idx = 0
saved = 0

for _ in tqdm(range(total), desc='нарезка'):
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % FRAME_STEP == 0:
        cv2.imwrite(f'{FRAMES_DIR}/frame_{frame_idx:06d}.jpg', frame)
        saved += 1
    frame_idx += 1

cap.release()
print(f'сохранено {saved} кадров в {FRAMES_DIR}')

frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))

## ПЗ 3 — Распознавание субтитров

Базовый код из ПЗ 3 + оптимизация: считаем короткий отпечаток нижней зоны кадра. Если отпечаток совпадает с предыдущим — субтитр не сменился, OCR пропускаем.

In [ ]:
import easyocr
import pandas as pd

reader = easyocr.Reader(['ru', 'en'], gpu=True)

SUBTITLE_FRACTION = 0.15


def dhash(img, size=8):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    r = cv2.resize(g, (size + 1, size))
    return (r[:, 1:] > r[:, :-1]).tobytes()


rows = []
prev_hash = None
skipped = 0

for fname in tqdm(frames, desc='ocr'):
    img = cv2.imread(f'{FRAMES_DIR}/{fname}')
    if img is None:
        continue
    h = img.shape[0]
    crop = img[int(h * (1 - SUBTITLE_FRACTION)):, :]

    h_now = dhash(crop)
    if h_now == prev_hash:
        skipped += 1
        continue
    prev_hash = h_now

    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    for _, text, prob in reader.readtext(gray):
        text = text.strip()
        if text and prob > 0.4:
            rows.append({'frame': fname, 'text': text, 'conf': round(prob, 3)})

df_ocr = pd.DataFrame(rows)
df_ocr.to_csv(f'{RESULTS_DIR}/ocr_results.csv', index=False, encoding='utf-8-sig')
print(f'пропущено повторов: {skipped} из {len(frames)}')
print(f'распознано строк: {len(df_ocr)}')

## ПЗ 4 — Транскрипция речи

In [ ]:
from moviepy.editor import VideoFileClip

audio_path = '/content/audio.wav'
clip = VideoFileClip(video_path)
clip.audio.write_audiofile(audio_path, verbose=False, logger=None)
clip.close()
print('аудио извлечено')

In [ ]:
import whisper

model = whisper.load_model('base')
result = model.transcribe(audio_path, verbose=False)

print('транскрипция готова')
print(result['text'][:500])

In [ ]:
rows = [
    {'start': round(s['start'], 2), 'end': round(s['end'], 2), 'text': s['text'].strip()}
    for s in result['segments']
]

df_whisper = pd.DataFrame(rows)
df_whisper.to_csv(f'{RESULTS_DIR}/whisper_transcript.csv', index=False, encoding='utf-8-sig')
print(f'сегментов: {len(df_whisper)}')
df_whisper.head(10)

## ПЗ 5 — Детекция объектов

In [ ]:
from ultralytics import YOLO

yolo = YOLO('yolov8n.pt')

In [ ]:
rows = []

for fname in tqdm(frames, desc='yolo'):
    res = yolo(f'{FRAMES_DIR}/{fname}', conf=CONF, verbose=False)
    for r in res:
        for box in r.boxes:
            rows.append({
                'frame':     fname,
                'frame_num': int(fname.split('_')[1].split('.')[0]),
                'class':     yolo.names[int(box.cls[0])],
                'conf':      round(float(box.conf[0]), 3),
                'x1': round(float(box.xyxy[0][0])),
                'y1': round(float(box.xyxy[0][1])),
                'x2': round(float(box.xyxy[0][2])),
                'y2': round(float(box.xyxy[0][3])),
            })

df_yolo = pd.DataFrame(rows)
df_yolo.to_csv(f'{RESULTS_DIR}/yolo_detections.csv', index=False)
print(f'детекций: {len(df_yolo)}')
df_yolo.head(10)

## ПЗ 6 — Классификация ResNet

In [ ]:
import json as _json
import torch
import urllib.request
from torchvision import models, transforms
from PIL import Image

urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json',
    '/content/imagenet_labels.json'
)
with open('/content/imagenet_labels.json') as f:
    LABELS = _json.load(f)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).to(device).eval()
print(f'resnet50 готов, устройство: {device}')

In [ ]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

rows = []

with torch.no_grad():
    for fname in tqdm(frames, desc='resnet'):
        img = Image.open(f'{FRAMES_DIR}/{fname}').convert('RGB')
        inp = transform(img).unsqueeze(0).to(device)
        probs = torch.softmax(resnet(inp), dim=1)[0]
        top5 = torch.topk(probs, 5)
        for rank, (idx, prob) in enumerate(zip(top5.indices.tolist(), top5.values.tolist()), 1):
            rows.append({'frame': fname, 'rank': rank, 'class': LABELS[idx], 'prob': round(prob, 4)})

df_resnet = pd.DataFrame(rows)
df_resnet.to_csv(f'{RESULTS_DIR}/resnet_classifications.csv', index=False)
print(f'обработано кадров: {len(frames)}')
df_resnet[df_resnet['rank'] == 1].head(10)

## ПЗ 7 — Описание сцен через LLM

In [ ]:
import requests
import time

resp = requests.get(f'{API_URL}/')
print('статус сервера:', resp.json())

In [ ]:
def describe_frame(image_path):
    with open(image_path, 'rb') as f:
        resp = requests.post(
            f'{API_URL}/describe',
            files={'file': (os.path.basename(image_path), f, 'image/jpeg')}
        )
    resp.raise_for_status()
    return resp.json()

STEP  = max(1, len(frames) // 20)
BATCH = frames[::STEP][:20]

print(f'модель: nvidia/nemotron-nano-12b-v2-vl:free (через OpenRouter)')
print(f'обрабатываем {len(BATCH)} кадров с паузой {PAUSE} сек')

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_frame(fname):
    try:
        res = describe_frame(f'{FRAMES_DIR}/{fname}')
        res['frame'] = fname
        return res
    except Exception as e:
        return {'frame': fname, 'objects': 'ошибка', 'error': str(e)}

results = []
with ThreadPoolExecutor(max_workers=5) as ex:
    futures = [ex.submit(process_frame, f) for f in BATCH]
    for fut in tqdm(as_completed(futures), total=len(BATCH), desc='openrouter'):
        res = fut.result()
        results.append(res)
        print(f'{res["frame"]}: {res.get("objects", "")[:80]}')

with open(f'{RESULTS_DIR}/llm_detections.json', 'w', encoding='utf-8') as f:
    _json.dump(results, f, ensure_ascii=False, indent=2)

print(f'\nобработано: {len(results)} кадров')

## ПЗ 8 — Постобработка и итоговый отчёт

In [ ]:
from rapidfuzz import fuzz, process
from collections import Counter

df_ocr = pd.read_csv(f'{RESULTS_DIR}/ocr_results.csv')
print(f'строк до дедупликации: {len(df_ocr)}')

def deduplicate(texts, threshold=85):
    unique = []
    for t in texts:
        if not unique:
            unique.append(t)
            continue
        best = process.extractOne(t, unique, scorer=fuzz.ratio)
        if best is None or best[1] < threshold:
            unique.append(t)
    return unique

unique = deduplicate(df_ocr['text'].dropna().tolist())
print(f'строк после дедупликации: {len(unique)}')

pd.DataFrame({'text': unique}).to_csv(f'{RESULTS_DIR}/ocr_dedup.csv', index=False, encoding='utf-8-sig')

In [ ]:
df_det = pd.read_csv(f'{RESULTS_DIR}/yolo_detections.csv')

def merge_detections(group):
    group = group.sort_values('frame_num').reset_index(drop=True)
    events = []
    start = group.iloc[0]
    prev  = start['frame_num']
    for _, row in group.iloc[1:].iterrows():
        if row['frame_num'] - prev > WINDOW:
            events.append({
                'class':       start['class'],
                'start_frame': start['frame_num'],
                'end_frame':   prev,
                'avg_conf':    round(group['conf'].mean(), 3),
            })
            start = row
        prev = row['frame_num']
    events.append({
        'class':       start['class'],
        'start_frame': start['frame_num'],
        'end_frame':   prev,
        'avg_conf':    round(group['conf'].mean(), 3),
    })
    return pd.DataFrame(events)

df_merged = (df_det
    .groupby('class', group_keys=False)
    .apply(merge_detections)
    .sort_values('start_frame')
    .reset_index(drop=True))

df_merged.to_csv(f'{RESULTS_DIR}/yolo_merged.csv', index=False)
print(f'детекций: {len(df_det)} -> событий: {len(df_merged)}')

In [ ]:
import json

with open(f'{RESULTS_DIR}/llm_detections.json', encoding='utf-8') as f:
    llm_data = json.load(f)

all_objects = []
for item in llm_data:
    objects = item.get('objects', '')
    if objects and objects != 'ошибка':
        for obj in objects.split(','):
            obj = obj.strip().lower()
            if obj:
                all_objects.append(obj)

counter = Counter(all_objects)
print('топ-15 объектов из LLM:')
for obj, cnt in counter.most_common(15):
    print(f'  {obj}: {cnt}')

In [ ]:
report = {
    'video_url':   VIDEO_URL,
    'video_title': raw_title,
    'folder':      VIDEO_DIR,
    'ocr': {
        'total_texts':  len(df_ocr),
        'unique_texts': len(unique),
        'dedup_ratio':  round(len(unique) / max(len(df_ocr), 1), 3),
    },
    'yolo': {
        'total_detections': len(df_det),
        'merged_events':    len(df_merged),
        'classes':          df_merged['class'].unique().tolist(),
    },
    'llm': {
        'frames_processed': len(llm_data),
        'model':            'nvidia/nemotron-nano-12b-v2-vl:free (OpenRouter)',
        'top_objects':      [obj for obj, _ in counter.most_common(10)],
    }
}

with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print('=== ФИНАЛЬНЫЙ ОТЧЁТ ===')
print(f"OCR: {report['ocr']['total_texts']} строк → {report['ocr']['unique_texts']} уникальных")
print(f"YOLO: {report['yolo']['total_detections']} детекций → {report['yolo']['merged_events']} событий")
print(f"LLM: {report['llm']['frames_processed']} кадров")
print(f"\nрезультаты: {VIDEO_DIR}")